# Save Ground-Truth ECG Signals as `.mat` Files

**What this notebook does:**
- Walks every PTB-XL `.hea` record under `DATA_DIR`
- Loads all 12 leads with `load_ecg()` (case-insensitive)
- Preprocesses with `preprocess_data()` (HP 0.5 Hz + LP 80 Hz, zero-phase)
- Saves a `.mat` file under `SAVE_DIR` mirroring the PTB-XL subfolder structure

```
Input  : records500/00000/00001_hr.hea   (PTB-XL)
Output : gt_clean/00000/00001_hr.mat
```

`.mat` contents:
| Field | Shape | Description |
|-------|-------|-------------|
| `ecg` | `(N, 12)` float64 | preprocessed signal |
| `lead_names` | `(1, 12)` cell | lead name strings |
| `record_name` | char | e.g. `'00001_hr'` |
| `fs` | scalar | sampling rate (500) |

## 1. Imports & Configuration

In [3]:
import os
import numpy as np
import scipy.io as sio
from pathlib import Path
from tqdm import tqdm
from scipy.signal import filtfilt

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR = "/Users/sinahassannia/Desktop/ptbxl_dataset/records500"
SAVE_DIR = "/Users/sinahassannia/Desktop/ecg_lead_cover2/gt_clean"

# Lead order — must stay consistent with the rest of the pipeline
ALL_LEADS = ['I', 'II', 'III', 'AVR', 'AVL', 'AVF',
             'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Input  : {DATA_DIR}")
print(f"Output : {SAVE_DIR}")
print(f"Leads  : {ALL_LEADS}")

Input  : /Users/sinahassannia/Desktop/ptbxl_dataset/records500
Output : /Users/sinahassannia/Desktop/ecg_lead_cover2/gt_clean
Leads  : ['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']


## 2. Pipeline Utilities — exact copies from `ecg_utils.py`

In [4]:
import wfdb

def load_ecg(record_path,
             selected_leads=['I', 'II', 'III', 'AVR', 'AVL', 'AVF',
                             'V1', 'V2', 'V3', 'V4', 'V5', 'V6']):
    """
    Load ECG record with flexible, case-insensitive lead name matching.

    Returns
    -------
    signal : (N, num_leads) float64 ndarray
    record : wfdb record object  (record.fs gives the sampling rate)
    """
    if record_path.endswith('.hea'):
        record_path = record_path[:-4]

    record = wfdb.rdrecord(record_path)
    signal = record.p_signal

    if selected_leads is not None:
        sig_names = list(record.sig_name)

        # case-insensitive index map
        available = {name.lower(): (name, idx)
                     for idx, name in enumerate(sig_names)}

        lead_indices, found_leads = [], []
        for req in selected_leads:
            key = req.lower()
            if key in available:
                actual, idx = available[key]
                lead_indices.append(idx)
                found_leads.append(actual)
            else:
                print(f"⚠️  Lead '{req}' not found. Available: {sig_names}")

        if not lead_indices:
            print("ERROR: No requested leads found!")
            return None, None

        signal = signal[:, lead_indices]
        record.sig_name = found_leads

    return signal, record



## 3. Discover All `.hea` Records

In [5]:
hea_files = sorted(Path(DATA_DIR).rglob('*.hea'))
print(f"Found {len(hea_files):,} .hea records under {DATA_DIR}")

# Preview
for p in hea_files[:5]:
    print(' ', p)

Found 21,837 .hea records under /Users/sinahassannia/Desktop/ptbxl_dataset/records500
  /Users/sinahassannia/Desktop/ptbxl_dataset/records500/00000/00001_hr.hea
  /Users/sinahassannia/Desktop/ptbxl_dataset/records500/00000/00002_hr.hea
  /Users/sinahassannia/Desktop/ptbxl_dataset/records500/00000/00003_hr.hea
  /Users/sinahassannia/Desktop/ptbxl_dataset/records500/00000/00004_hr.hea
  /Users/sinahassannia/Desktop/ptbxl_dataset/records500/00000/00005_hr.hea


## 4. Single-Record Worker

In [6]:
def process_and_save(hea_path: Path,
                     data_dir: str,
                     save_dir: str,
                     leads: list = ALL_LEADS) -> bool:
    """
    Load one PTB-XL record, preprocess, and save as .mat.

    Parameters
    ----------
    hea_path : Path to the .hea file
    data_dir : root DATA_DIR (used to compute the relative subfolder)
    save_dir : root SAVE_DIR for output .mat files
    leads    : lead names to extract (case-insensitive)

    Returns
    -------
    True on success, False on any error.

    .mat fields
    -----------
    ecg         — (N, num_leads) float64, preprocessed
    lead_names  — (1, num_leads) cell array of strings
    record_name — char, e.g. '00001_hr'
    fs          — scalar, sampling rate in Hz
    """
    rec_path    = str(hea_path)                        # full path with .hea
    record_name = hea_path.stem                        # e.g. '00001_hr'

    # ── 1. Load ───────────────────────────────────────────────────────────
    try:
        signal_pp, record = load_ecg(rec_path, selected_leads=leads)
        if signal_pp is None:
            return False
        fs = record.fs
    except Exception as e:
        print(f"  [LOAD ERROR]  {rec_path}: {e}")
        return False

    
    # ── 3. Build output path (mirror subfolder structure) ─────────────────
    rel_folder = os.path.relpath(str(hea_path.parent), data_dir)
    out_dir    = os.path.join(save_dir, rel_folder)
    os.makedirs(out_dir, exist_ok=True)
    mat_path   = os.path.join(out_dir, f"{record_name}.mat")

    # ── 4. Save .mat ──────────────────────────────────────────────────────
    # scipy.io.savemat needs lead_names as a (1, N) object array for MATLAB cell
    actual_leads   = list(record.sig_name)             # names actually found
    lead_names_cell = np.array(actual_leads, dtype=object).reshape(1, -1)

    mat_dict = {
        'ecg'         : signal_pp.astype(np.float64),
        'lead_names'  : lead_names_cell,
        'record_name' : record_name,
        'fs'          : int(fs),
    }

    try:
        sio.savemat(mat_path, mat_dict)
    except Exception as e:
        print(f"  [SAVE ERROR]  {mat_path}: {e}")
        return False

    return True


print("process_and_save() defined.")

process_and_save() defined.


## 5. Single-File Test

In [7]:
test_hea = hea_files[0]
print(f"Testing on: {test_hea}")

ok = process_and_save(test_hea, DATA_DIR, SAVE_DIR)

if ok:
    # ── Round-trip verification ────────────────────────────────────────────
    rel_folder  = os.path.relpath(str(test_hea.parent), DATA_DIR)
    mat_path    = os.path.join(SAVE_DIR, rel_folder, f"{test_hea.stem}.mat")
    loaded      = sio.loadmat(mat_path, squeeze_me=True)

    print(f"\n.mat keys      : {[k for k in loaded if not k.startswith('_')]}")
    print(f"ecg shape      : {loaded['ecg'].shape}   (samples × leads)")
    print(f"ecg dtype      : {loaded['ecg'].dtype}")
    print(f"lead_names     : {list(loaded['lead_names'])}")
    print(f"record_name    : {loaded['record_name']}")
    print(f"fs             : {loaded['fs']} Hz")
    print(f"value range    : [{loaded['ecg'].min():.4f}, {loaded['ecg'].max():.4f}] mV")
    print("\n✅ Single-file test passed.")
else:
    print("❌ Test failed — check errors above.")

Testing on: /Users/sinahassannia/Desktop/ptbxl_dataset/records500/00000/00001_hr.hea

.mat keys      : ['ecg', 'lead_names', 'record_name', 'fs']
ecg shape      : (5000, 12)   (samples × leads)
ecg dtype      : float64
lead_names     : ['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
record_name    : 00001_hr
fs             : 500 Hz
value range    : [-1.4440, 0.6900] mV

✅ Single-file test passed.


## 6. Batch Convert — Full Dataset

In [8]:
n_ok   = 0
n_fail = 0
failed = []

for hea_path in tqdm(hea_files, desc="Saving GT .mat", unit="file"):
    ok = process_and_save(hea_path, DATA_DIR, SAVE_DIR, leads=ALL_LEADS)
    if ok:
        n_ok += 1
    else:
        n_fail += 1
        failed.append(str(hea_path))

print(f"\n✅ Saved  : {n_ok:,}")
print(f"❌ Failed : {n_fail:,}")
if failed:
    print("\nFailed files:")
    for f in failed:
        print(' ', f)

Saving GT .mat: 100%|██████████| 21837/21837 [00:44<00:00, 495.76file/s]


✅ Saved  : 21,837
❌ Failed : 0


## 7. Verify Output Structure

In [9]:
mat_files = sorted(Path(SAVE_DIR).rglob('*.mat'))
print(f"Total .mat files written : {len(mat_files):,}")
print(f"Total .hea files found   : {len(hea_files):,}")

print("\nSample output paths:")
for p in mat_files[:5]:
    print(' ', p)

Total .mat files written : 21,837
Total .hea files found   : 21,837

Sample output paths:
  /Users/sinahassannia/Desktop/ecg_lead_cover2/gt_clean/00000/00001_hr.mat
  /Users/sinahassannia/Desktop/ecg_lead_cover2/gt_clean/00000/00002_hr.mat
  /Users/sinahassannia/Desktop/ecg_lead_cover2/gt_clean/00000/00003_hr.mat
  /Users/sinahassannia/Desktop/ecg_lead_cover2/gt_clean/00000/00004_hr.mat
  /Users/sinahassannia/Desktop/ecg_lead_cover2/gt_clean/00000/00005_hr.mat


## 8. Quick Sanity Check — Random Sample

In [10]:
import random

sample = str(random.choice(mat_files))
data   = sio.loadmat(sample, squeeze_me=True)

print(f"File        : {sample}")
print(f"record_name : {data['record_name']}")
print(f"ecg shape   : {data['ecg'].shape}   (samples × leads)")
print(f"ecg dtype   : {data['ecg'].dtype}")
print(f"fs          : {data['fs']} Hz")
print(f"lead_names  : {list(data['lead_names'])}")
print(f"value range : [{data['ecg'].min():.4f}, {data['ecg'].max():.4f}] mV")

File        : /Users/sinahassannia/Desktop/ecg_lead_cover2/gt_clean/19000/19182_hr.mat
record_name : 19182_hr
ecg shape   : (5000, 12)   (samples × leads)
ecg dtype   : float64
fs          : 500 Hz
lead_names  : ['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
value range : [-1.9260, 5.2150] mV


## How to Load in Python

```python
import scipy.io as sio

data       = sio.loadmat('gt_clean/00000/00001_hr.mat', squeeze_me=True)
ecg        = data['ecg']               # (5000, 12) float64 — preprocessed
lead_names = list(data['lead_names'])  # ['I', 'II', 'III', ...]
fs         = int(data['fs'])           # 500

# Extract Lead II (index 1)
lead_II = ecg[:, lead_names.index('II')]  # or whichever index 'II' is
```

## How to Load in MATLAB

```matlab
data  = load('gt_clean/00000/00001_hr.mat');
ecg   = data.ecg;          % (5000 × 12) double — preprocessed
leads = data.lead_names;   % 1×12 cell
fs    = data.fs;           % 500
```